# ALIGN Tutorial Notebook: CHILDES

This notebook provides an introduction to **ALIGN**, 
a tool for quantifying multi-level linguistic similarity 
between speakers, using parent-child transcript data from 
the Kuczaj Corpus 
(https://childes.talkbank.org/access/Eng-NA/Kuczaj.html).

This method was introduced in "ALIGN: Analyzing Linguistic Interactions with Generalizable techNiques" (Duran, Paxton, & Fusaroli, 2019).

## Tutorial Overview

While many studies of interpersonal linguistic alignment
compare alignment between different dyads across different
conditions (i.e., typically a within- or between-dyads
design in which each dyad contributes only one or two 
conversations), there may also be interest in understanding
longer-scale temporal dynamics *within* a given dyad.
This tutorial provides an example of how ALIGN may be used
to just that end: analyzing how a single dyad's multilevel
alignment changes across different conversations held
at different points over a longer range of time.

To do so, this tutorial walks users through an analysis of
conversations from a single English corpus from the CHILDES 
database  (MacWhinney, 2000)---specifically, Kuczaj’s Abe 
corpus (Kuczaj, 1976), used under a Creative Commons 
Attribution-ShareAlike 3.0 Unported License (see GitHub
repository or `data/CHILDES` directory for license). 
We analyze the last 20 conversations in the corpus in order
to explore how ALIGN can be used to track multi-level
linguistic alignment between a parent and child over time,
which may be of interest to developmental language
researchers. Specifically, we explore how alignment between a parent
and a child changes over a brief span of developmental
trajectory.

Data for this tutorial are shipped with the `align`
package on PyPI (https://pypi.python.org/pypi/align) and GitHub
(https://github.com/nickduran/align-linguistic-alignment/).

***

## Table of Contents

* [Getting Started](#Getting-Started)
    * [Prerequisites](#Prerequisites)
    * [Preparing input data](#Preparing-input-data)
    * [Filename conventions](#Filename-conventions)
    * [Highest-level functions](#Highest-level-functions)
* [Setup](#Setup)
    * [Import libraries](#Import-libraries)
    * [Specify ALIGN path settings](#Specify-ALIGN-path-settings)
* [Phase 1: Prepare transcripts](#Phase-1:-Prepare-transcripts)
    * [Preparation settings](#Preparation-settings)
    * [Run preparation phase](#Run-preparation-phase)
* [Phase 2: Calculate alignment](#Phase-2:-Calculate-alignment)
    * [For real data: Alignment calculation settings](#For-real-data:-Alignment-calculation-settings)
    * [For real data: Run alignment calculation](#For-real-data:-Run-alignment-calculation)
    * [For surrogate data: Alignment calculation settings](#For-surrogate-data:-Alignment-calculation-settings)
    * [For surrogate data: Run alignment calculation](#For-surrogate-data:-Run-alignment-calculation)
* [ALIGN output overview](#ALIGN-output-overview)
    * [Speed calculations](#Speed-calculations)
    * [Printouts!](#Printouts!)

***

# Getting Started

## Preparing input data

* Each input text file needs to contain a single conversation organized in an `N x 2` matrix
    * Text file must be tab-delimited.
* Each row must correspond to a single conversational turn from a speaker.
    * Rows must be temporally ordered based on their occurrence in the conversation.
    * Rows must alternate between speakers.
* Speaker identifier and content for each turn are divided across two columns.
    * Column 1 must have the header `participant`.
        * Each cell specifies the speaker.
        * Each speaker must have a unique label (e.g., `P1` and `P2`, `0` and `1`).
    * Column 2 must have the header `content`.
        * Each cell corresponds to the transcribed utterance from the speaker.
        * Each cell must end with a newline character: `\n`
* See `examples` directory in Github repository for an example

## Filename conventions

* Each conversation text file must be regularly formatted, including a prefix for dyad and a prefix for conversation prior to the identifier for each that are separated by a unique character. By default, ALIGN looks for patterns that follow this convention: `dyad1-condA.txt`
    * However, users may choose to include any label for dyad or condition so long as the two labels are distinct from one another and are not subsets of any possible dyad or condition labels. Users may also use any character as a separator so long as it does not occur anywhere else in the filename.
    * The chosen file format **must** be used when saving **all** files for this analysis.

## Highest-level functions

Given appropriately prepared transcript files, ALIGN can be run in 3 high-level functions:

**`prepare_transcripts`**: Pre-process each standardized 
conversation, checking it conforms to the requirements. 
Each utterance is tokenized and lemmatized and has 
POS tags added.

**`calculate_alignment`**: Generates turn-level and 
conversation-level alignment scores (lexical, 
conceptual, and syntactic) across a range of 
*n*-gram sequences.

**`calculate_baseline_alignment`**: Generate a surrogate corpus
and run alignment analysis (using identical specifications 
from `calculate_alignment`) on it to produce a baseline.

***

# Setup

## Import libraries

Install ALIGN if you have not already.

In [1]:
import sys
!{sys.executable} -m pip install align


[notice] A new release of pip is available: 22.0.4 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Import packages we'll need to run ALIGN.

In [2]:
import align, os
import pandas as pd

Import `time` so that we can get a sense of how
long the ALIGN pipeline takes.

In [3]:
import time

Import `warnings` to flag us if required files aren't provided.

In [4]:
import warnings

## Install additional NTLK packages

Download some addition `nltk` packages for `align` to work.

In [5]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to C:\Users\Hardik
[nltk_data]     Maisuria\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Hardik Maisuria\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to C:\Users\Hardik
[nltk_data]     Maisuria\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Hardik
[nltk_data]     Maisuria\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

## Specify ALIGN path settings

ALIGN will need to know where the raw transcripts are stored, where to store the processed data, and where to read in any additional files needed for optional ALIGN parameters.

### Required directories

For the sake of this tutorial, specify a base path that will serve as our jumping-off point for our saved data. All of the CHILDES data and shipped data will be called from the package directory.

**`BASE_PATH`**: Containing directory for this tutorial.

In [6]:
BASE_PATH = os.getcwd()

**`CHILDES_EXAMPLE`**: Subdirectories for output and other
files for this tutorial. (We'll create a default directory
if one doesn't already exist.)

In [7]:
CHILDES_EXAMPLE = os.path.join(BASE_PATH,
                              'CHILDES/')

In [8]:
if not os.path.exists(CHILDES_EXAMPLE):
    os.makedirs(CHILDES_EXAMPLE)

**`TRANSCRIPTS`**: Path to raw transcript files. Automatically provided by `align`.

In [9]:
TRANSCRIPTS = align.datasets.CHILDES_directory

**`PREPPED_TRANSCRIPTS`**: Set variable for folder name 
(as string) for relative location of folder into which 
prepared transcript files will be saved. (We'll create
a default directory if one doesn't already exist.)

In [10]:
PREPPED_TRANSCRIPTS = os.path.join(CHILDES_EXAMPLE,
                                   'childes-prepped/')

In [11]:
if not os.path.exists(PREPPED_TRANSCRIPTS):
    os.makedirs(PREPPED_TRANSCRIPTS)

**`ANALYSIS_READY`**: Set variable for folder name 
(as string) for relative location of folder into 
which analysis-ready dataframe files will be saved.
(We'll create a default directory if one doesn't
already exist.)

In [12]:
ANALYSIS_READY = os.path.join(CHILDES_EXAMPLE,
                              'childes-analysis/')

In [13]:
if not os.path.exists(ANALYSIS_READY):
    os.makedirs(ANALYSIS_READY)

**`SURROGATE_TRANSCRIPTS`**: Set variable for folder name 
(as string) for relative location of folder into which all
prepared surrogate transcript files will be saved. (We'll
create a default directory if one doesn't already exist.)

In [14]:
SURROGATE_TRANSCRIPTS = os.path.join(CHILDES_EXAMPLE,
                                     'childes-surrogate/')

In [15]:
if not os.path.exists(SURROGATE_TRANSCRIPTS):
    os.makedirs(SURROGATE_TRANSCRIPTS)

### Paths for optional parameters

**`OPTIONAL_PATHS`**: If using Stanford POS tagger or
pretrained vectors, the path to these files. If these
files are provided in other locations, be sure to
change the file paths for them. (We'll create a default
directory if one doesn't already exist.)

In [16]:
OPTIONAL_PATHS = os.path.join(CHILDES_EXAMPLE,
                             'optional_directories/')

In [17]:
if not os.path.exists(OPTIONAL_PATHS):
    os.makedirs(OPTIONAL_PATHS)

#### Stanford POS Tagger

The Stanford POS tagger **will not be used** by 
default in this example. However, you may use them
by uncommenting and providing the requested file 
paths in the cells in this section and then changing 
the relevant parameters in the ALIGN calls below.

If desired, we could use the Standford part-of-speech 
tagger along with the Penn part-of-speech tagger
(which is always used in ALIGN). To do so, the files
will need to be downloaded separately: 
https://nlp.stanford.edu/software/tagger.shtml#Download

**`STANFORD_POS_PATH`**: If using Stanford POS tagger
with the Penn POS tagger, path to Stanford directory.

In [19]:
# STANFORD_POS_PATH = os.path.join(OPTIONAL_PATHS,
#                                  'stanford-postagger-full-2018-10-16/')

In [20]:
# if os.path.exists(STANFORD_POS_PATH) == False:
#     warnings.warn('Stanford POS directory not found at the specified '
#                       'location. Please update the file path with '
#                       'the folder that can be directly downloaded here: '
#                       'https://nlp.stanford.edu/software/stanford-postagger-full-2018-10-16.zip '
#                       '- Alternatively, comment out the '
#                       '`STANFORD_POS_PATH` information.')

**`STANFORD_LANGUAGE`**: If using Stanford tagger,
set language model to be used for POS tagging.

In [21]:
# STANFORD_LANGUAGE = os.path.join('models/english-left3words-distsim.tagger')

In [22]:
# if os.path.exists(STANFORD_POS_PATH + STANFORD_LANGUAGE) == False:
#     warnings.warn('Stanford tagger language not found at the specified '
#                       'location. Please update the file path or comment '
#                       'out the `STANFORD_POS_PATH` information.')

#### Google News pretrained vectors

The Google News pretrained vectors **will be used**
by default in this example. The file is available for
download here: https://code.google.com/archive/p/word2vec/

If desired, researchers may choose to read in pretrained
`word2vec` vectors rather than creating a semantic space
from the corpus provided. This may be especially useful 
for small corpora (i.e., fewer than 30k unique words),
although the choice of semantic space corpus should be
made with careful consideration about the nature of the
linguistic context (for further discussion, see Duran, 
Paxton, & Fusaroli, *submitted*).

**`PRETRAINED_INPUT_FILE`**: If using pretrained vectors, path
to pretrained vector files. You may choose to download the file
directly to this path or change the path to a different one.

In [18]:
PRETRAINED_INPUT_FILE = os.path.join(OPTIONAL_PATHS,
                            'GoogleNews-vectors-negative300.bin')

In [19]:
if os.path.exists(PRETRAINED_INPUT_FILE) == False:
    warnings.warn('Google News vector not found at the specified '
                      'location. Please update the file path with '
                      'the .bin file that can be accessed here: '
                      'https://code.google.com/archive/p/word2vec/ '
                      '- Alternatively, comment out the `PRETRAINED_INPUT_FILE` information')

C:\Users\Hardik Maisuria\AppData\Local\Temp\ipykernel_5160\642821443.py:2: UserWarning: Google News vector not found at the specified location. Please update the file path with the .bin file that can be accessed here: https://code.google.com/archive/p/word2vec/ - Alternatively, comment out the `PRETRAINED_INPUT_FILE` information
  warnings.warn('Google News vector not found at the specified '


***

# Phase 1: Prepare transcripts

In Phase 1, we take our raw transcripts and get them ready
for later ALIGN analysis.

## Preparation settings

There are a number of parameters that we can set for the
`prepare_transcripts()` function:

In [20]:
print(align.prepare_transcripts.__doc__)


    Prepare transcripts for similarity analysis.

    Given individual .txt files of conversations,
    return a completely prepared dataframe of transcribed
    conversations for later ALIGN analysis, including: text
    cleaning, merging adjacent turns, spell-checking,
    tokenization, lemmatization, and part-of-speech tagging.
    The output serve as the input for later ALIGN
    analysis.

    Parameters
    ----------
    input_files : str (directory name) or list of str (file names)
        Raw files to be cleaned. Behavior governed by `input_as_directory`
        parameter as well.

    output_file_directory : str
        Name of directory where output for individual conversations will be
        saved.

    run_spell_check : boolean, optional (default: True)
        Specify whether to run the spell-checking algorithm (True) or to 
        ignore it (False). 

    training_dictionary : str, optional (default: None)
        Specify whether to train the spell-checking dictionary

For the sake of this demonstration, we'll keep everything as
defaults. Among other parameters, this means that:
* any turns fewer than 2 words will be removed from the corpus
 (`minwords=2`),
* we'll be using regex to strip out any filler words
 (e.g., "uh," "um," "huh"; `use_filler_list=None`),
* if you like, you can ignore the regex option and supply additional filler words as `use_filler_list=["string1", "string2"]`
* moreover, if you like, you can include regex and supply your own filler words, but be sure to set `filler_regex_and_list=True`   
* we'll be using the Project Gutenberg corpus to create our 
 spell-checker algorithm (`training_dictionary=None`),
* we'll rely only on the Penn POS tagger 
 (`add_stanford_tags=False`), and
* our data will be saved both as individual conversation files
 and as a master dataframe of all conversation outputs
 (`save_concatenated_dataframe=True`).

## Run preparation phase

First, we prepare our transcripts by reading in individual `.txt`
files for each conversation, clean up undesired text and turns,
spell-check, tokenize, lemmatize, and add POS tags.

In [21]:
start_phase1 = time.time()

In [22]:
model_store = align.prepare_transcripts(
                        input_files=TRANSCRIPTS,
                        output_file_directory=PREPPED_TRANSCRIPTS,
                        run_spell_check=True,
                        minwords=2,
                        use_filler_list=None,
                        filler_regex_and_list=False,
                        training_dictionary=None,
                        add_stanford_tags=False,
                            ## if you want to run the Stanford POS tagger, be sure to set `add_stanford_tags` to `True` and uncomment the next two lines
                            # stanford_pos_path=STANFORD_POS_PATH,
                            # stanford_language_path=STANFORD_LANGUAGE,
                        save_concatenated_dataframe=True)

Processing: d:\ASU\ALIGN\alignFeatureRepoClone\src\align\semantic_alignment_venv\lib\site-packages\align\data/CHILDES\time191-cond1.txt
Processing: d:\ASU\ALIGN\alignFeatureRepoClone\src\align\semantic_alignment_venv\lib\site-packages\align\data/CHILDES\time192-cond1.txt
Processing: d:\ASU\ALIGN\alignFeatureRepoClone\src\align\semantic_alignment_venv\lib\site-packages\align\data/CHILDES\time193-cond1.txt
Processing: d:\ASU\ALIGN\alignFeatureRepoClone\src\align\semantic_alignment_venv\lib\site-packages\align\data/CHILDES\time194-cond1.txt
Processing: d:\ASU\ALIGN\alignFeatureRepoClone\src\align\semantic_alignment_venv\lib\site-packages\align\data/CHILDES\time195-cond1.txt
Processing: d:\ASU\ALIGN\alignFeatureRepoClone\src\align\semantic_alignment_venv\lib\site-packages\align\data/CHILDES\time196-cond1.txt
Processing: d:\ASU\ALIGN\alignFeatureRepoClone\src\align\semantic_alignment_venv\lib\site-packages\align\data/CHILDES\time197-cond1.txt
Processing: d:\ASU\ALIGN\alignFeatureRepoClone\s

In [23]:
end_phase1 = time.time()

# Phase 2: Calculate Semantic alignment

In [24]:
from semantic_alignment import semantic_alignment

d:\ASU\ALIGN\alignFeatureRepoClone\src\align\semantic_alignment_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [27]:
# Define folder paths

output_file_directory = "outputW2V"
# Attempts to load a pre-existing embedding cache from a file (bert_embedding_cache.pkl).
BERT_embedding_cache_path = "bert_embedding_cache.pkl"
result_df = semantic_alignment(PREPPED_TRANSCRIPTS, 
                                         output_file_directory, 
                                         BERT_embedding_cache_path,
                                         use_W2V = True, 
                                         use_BERT=True
                                         )

Local cache directory: d:\ASU\ALIGN\alignFeatureRepoClone\examples\gensim-data
Model word2vec-google-news-300 already exists at: d:\ASU\ALIGN\alignFeatureRepoClone\examples\gensim-data\word2vec-google-news-300
Model glove-twitter-200 already exists at: d:\ASU\ALIGN\alignFeatureRepoClone\examples\gensim-data\glove-twitter-200
Word2Vec Google News model loaded successfully.


Processing files for BERT: 100%|██████████| 20/20 [00:01<00:00, 11.78it/s]


***

# ALIGN output overview

In [28]:
result_df.head()

,participant,content,token,lemma,tagged_token,tagged_lemma,file,content1,content2,token1,...,lemma,tagged_token,tagged_lemma,file,content1,content2,utter_order,content1_embedding_BERT,content2_embedding_BERT,content1_embedding_BERT_content2_embedding_BERT_cosine_similarity
0,kid,my legs are too tired for to walk all the way ...,"[my, too, for, walk, all, way, over, there]","[my, leg, too, for, walk, all, way, over, there]","[(my, PRP$), (legs, NNS), (are, VBP), (too, RB...","[(my, PRP$), (leg, NN), (be, VB), (too, RB), (...",time191-cond1.txt,my legs are too tired for to walk all the way ...,what are you thinking about well do you want ...,"[my, too, for, walk, all, way, over, there]",...,"['my', 'leg', 'be', 'too', 'tired', 'for', 'to...","[('my', 'PRP$'), ('legs', 'NNS'), ('are', 'VBP...","[('my', 'PRP$'), ('leg', 'NN'), ('be', 'VB'), ...",time191-cond1.txt,my legs are too tired for to walk all the way ...,what are you thinking about well do you want ...,kid cgv,"[[0.1197973, -0.009500369, 0.032547843, -0.130...","[[0.5135089, -0.15353827, 0.3197633, -0.132093...",0.669601
1,cgv,what are you thinking about well do you want ...,"[about, well, want, take, nap, how, come, want...","[think, about, well, want, take, nap, how, com...","[(what, WP), (are, VBP), (you, PRP), (thinking...","[(what, WP), (be, VB), (you, PRP), (think, VBP...",time191-cond1.txt,what are you thinking about well do you want ...,okay do you,"[about, well, want, take, nap, how, come, want...",...,"['what', 'be', 'you', 'think', 'about', 'well'...","[('what', 'WP'), ('are', 'VBP'), ('you', 'PRP'...","[('what', 'WP'), ('be', 'VB'), ('you', 'PRP'),...",time191-cond1.txt,what are you thinking about well do you want ...,okay do you,cgv kid,"[[0.5135089, -0.15353827, 0.3197633, -0.132093...","[[0.40721783, -0.21188489, 0.1909192, -0.00547...",0.539146
2,kid,okay do you,[okay],[okay],"[(okay, RB), (do, VBP), (you, PRP)]","[(okay, RB), (do, VBP), (you, PRP)]",time191-cond1.txt,okay do you,yeah wow,[okay],...,"['okay', 'do', 'you']","[('okay', 'RB'), ('do', 'VBP'), ('you', 'PRP')]","[('okay', 'RB'), ('do', 'VBP'), ('you', 'PRP')]",time191-cond1.txt,okay do you,yeah wow,kid cgv,"[[0.40721783, -0.21188489, 0.1909192, -0.00547...","[[0.3745891, 0.10791623, 0.15194824, 0.1889031...",0.649014
3,cgv,yeah wow,"[yeah, now]","[yeah, now]","[(yeah, RB), (now, RB)]","[(yeah, RB), (now, RB)]",time191-cond1.txt,yeah wow,doesn't it float good,"[yeah, now]",...,"['yeah', 'now']","[('yeah', 'RB'), ('now', 'RB')]","[('yeah', 'RB'), ('now', 'RB')]",time191-cond1.txt,yeah wow,doesn't it float good,cgv kid,"[[0.3745891, 0.10791623, 0.15194824, 0.1889031...","[[0.13908246, 0.09258423, 0.20989531, 0.195118...",0.528134
4,kid,doesn't it float good,[good],[good],"[(does, VBZ), (not, RB), (it, PRP), (float, VB...","[(do, VBP), (not, RB), (it, PRP), (float, VBZ)...",time191-cond1.txt,doesn't it float good,it sure does,[good],...,"['do', 'not', 'it', 'float', 'good']","[('does', 'VBZ'), ('not', 'RB'), ('it', 'PRP')...","[('do', 'VBP'), ('not', 'RB'), ('it', 'PRP'), ...",time191-cond1.txt,doesn't it float good,it sure does,kid cgv,"[[0.13908246, 0.09258423, 0.20989531, 0.195118...","[[0.41619697, 0.110191725, 0.542153, 0.2311180...",0.648298
